In [ ]:
!pip install -U langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 32.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [ ]:
!pip install langchain

In [ ]:
google_api_key = "Add your Google API key here"

In [ ]:
from langchain_core.prompts import PromptTemplate,load_prompt
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)


In [ ]:
## simple chain

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)


model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)


parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)

# chain.get_graph().print_ascii()

Here are 5 interesting facts about cricket:

### 1. The First International Sporting Match in History Was Cricket (USA vs. Canada)
While cricket is often associated with the UK, Australia, and South Asia, the very first international match of *any* sport took place in New York City in 1844. Canada played against the United States at the St George's Cricket Club in Manhattan. Canada won the match by 23 runs in front of a crowd of around 10,000 spectators. 

### 2. The Longest Test Match Ended Because a Boat Was Leaving
In March 1939, England and South Africa played a "Timeless Test" in Durban—meaning it was meant to be played until one team won, regardless of how many days it took. After **12 days** (9 days of actual play), the game was officially abandoned as a draw. Why? The English team was going to miss their ship back home if they didn't leave immediately. Over the course of the match, 1,981 overs were bowled and 1,983 runs were scored.

### 3. Sir Donald Bradman Needed Only 4 Runs

In [ ]:
## sequential chain

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic': 'Unemployment in India'})

print(result)

# chain.get_graph().print_ascii()

Here is a 5-pointer summary based strictly on the provided report:

* **The Demographic Paradox:** Although over 65% of India's population is under 35, low official unemployment figures (~3.1%–3.2% according to PLFS) mask severe structural issues, including widespread underemployment, high informal sector reliance (~82%–85%), and disguised unemployment in agriculture.
* **Severe Youth and Educated Unemployment:** Unemployment disproportionately impacts young and educated demographics; youth unemployment reaches 10%–16% nationwide (and over 20% in urban areas), with university graduates facing unemployment rates nearly nine times higher than illiterate individuals.
* **Structural Causes of "Jobless Growth":** India's economic path skipped broad-based, labor-intensive manufacturing to move directly from agriculture to high-skilled services. Consequently, manufacturing has stagnated at 15%–17% of GDP, agriculture remains over-dependent (absorbing over 42% of labor for under 18% of GDP), a

In [ ]:
# parallel chains


from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
model1 = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

model2 = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print(result)

#chain.get_graph().print_ascii()


Here is the merged document containing both the study notes and the practice quiz.

---

# Support Vector Machines (SVMs): Study Notes & Practice Quiz

## **Part 1: Study Notes**

### **Overview**
* **Definition:** Supervised learning methods used for classification, regression, and outlier detection.

---

### **Advantages**
* **High Dimensions:** Effective in high-dimensional spaces, even when the number of dimensions exceeds the number of samples.
* **Memory Efficient:** Uses a subset of training points (support vectors) in the decision function.
* **Versatile:** Supports various standard or custom Kernel functions.

---

### **Disadvantages**
* **Overfitting Risk:** Requires careful choice of Kernels and regularization terms if features far outnumber samples.
* **No Direct Probabilities:** Does not directly calculate probability estimates; calculating them requires expensive five-fold cross-validation.

---

### **Input & Performance (scikit-learn)**
* **Supported Inputs:** Accepts

In [ ]:
## conditional chains


from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",   # best free model
    google_api_key= google_api_key
)

parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

##chain.get_graph().print_ascii()